# Analyst-Ready SEC Filings RAG System

**Course:** AI Engineering  
**Professor:** Apostolos Filippas  
**Institution:** Fordham University  
**Team:** Joel Ebukatobi, Claudia Gisemba

This notebook builds a complete SEC 10-K Retrieval-Augmented Generation pipeline for analyst workflows.

In [ ]:
# Section 1: Setup & Configuration
import os
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv

from src.ingest import load_raw_filings, filing_stats
from src.chunk import parse_sections, section_coverage_stats, build_chunks, chunk_stats
from src.embed import embed_chunks, embedding_info
from src.retrieve import HybridRetriever
from src.generate import compare, generate_structured_output
from src.evaluate import build_test_set, run_retrieval_eval, ablation_table

load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    print("Warning: OPENAI_API_KEY is not set. Generation sections will fail until set.")
else:
    print("OpenAI API key loaded successfully.")

# Global constants for a reproducible project configuration.
CHUNK_SIZE = 800
CHUNK_OVERLAP = 200
TOP_K = 5
EMBED_MODEL = "all-MiniLM-L6-v2"
LLM_MODEL = "gpt-4o"
TICKERS = ["TSLA", "AAPL", "MSFT", "AMZN", "NVDA"]
YEAR_RANGE = (2018, 2023)
TARGET_SECTIONS = ["section_1a", "section_7", "section_3"]

Path("data").mkdir(parents=True, exist_ok=True)
print("Constants defined and environment loaded.")

## Section 2: Data Ingestion
Load 10-K filings from Hugging Face and normalize into a DataFrame with core metadata.

In [ ]:
df_filings = load_raw_filings(
    tickers=TICKERS,
    year_range=YEAR_RANGE,
    filing_type="10-K"
)

print("Raw filing DataFrame shape:", df_filings.shape)
display(df_filings.head(3))
display(filing_stats(df_filings).head(20))

## Section 3: Section Parsing
Extract Item 1A (Risk Factors), Item 7 (MD&A), and Item 3 (Legal Proceedings) from filing text.

In [ ]:
df_sections = parse_sections(df_filings, target_sections=TARGET_SECTIONS)
print("Parsed sections shape:", df_sections.shape)
display(df_sections.head(5))
display(section_coverage_stats(df_sections))

## Section 4: Section-Aware Chunking
Chunk each parsed section independently so chunks never cross section boundaries.

In [ ]:
all_chunks = build_chunks(
    df_sections,
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    cache_path="data/chunks.pkl",
    use_multiprocessing=True
)

print("Total chunk count:", len(all_chunks))
display(pd.DataFrame(all_chunks).head(3))
display(chunk_stats(all_chunks))

## Section 5: Embedding
Generate semantic vectors for every chunk and cache to disk.

In [ ]:
vectors, vector_metadata = embed_chunks(
    all_chunks,
    model_name=EMBED_MODEL,
    batch_size=256,
    cache_path="data/embeddings.pkl"
)

print("Vectors shape:", getattr(vectors, "shape", None))
print("Embedding info:", embedding_info(vectors))

## Section 6: Hybrid Retrieval
Combine FAISS semantic retrieval with BM25 lexical retrieval using weighted reranking.

In [ ]:
retriever = HybridRetriever(
    vectors=vectors,
    metadata=vector_metadata,
    embed_model_name=EMBED_MODEL,
    semantic_weight=0.6,
    lexical_weight=0.4
)

sample_queries = [
    "major macroeconomic risk factors",
    "changes in segment revenue drivers",
    "notable legal proceedings and litigation"
]

for q in sample_queries:
    results = retriever.retrieve(query=q, ticker="AAPL", year=2022, section_type="section_1a", top_k=TOP_K)
    print("\nQuery:", q)
    display(pd.DataFrame(results)[["chunk_id", "semantic_score", "bm25_score", "final_score"]].head(TOP_K))

## Section 7: Comparative Reasoning
Retrieve evidence from two filing years and ask GPT-4o for structured differences.

In [ ]:
comparison_result = compare(
    retriever=retriever,
    ticker="AAPL",
    section_type="section_1a",
    year_a=2020,
    year_b=2023,
    query="material risk changes over time",
    top_k=TOP_K,
    model=LLM_MODEL
)

print("Raw diff output:")
print(comparison_result.get("raw_diff", "")[:3000])

## Section 8: Structured Output Generation
Convert reasoning output into strict JSON schema and validate it with graceful error handling.

In [ ]:
structured_output = generate_structured_output(comparison_result, model=LLM_MODEL)

if "error" in structured_output:
    print("Structured output failed gracefully:")
    print(json.dumps(structured_output, indent=2))
else:
    print("Validated structured JSON output:")
    print(json.dumps(structured_output, indent=2)[:5000])

## Section 9: Evaluation
Evaluate retrieval and structured generation quality. Include ablation-style comparison table.

In [ ]:
test_set = build_test_set()
eval_results = run_retrieval_eval(retriever, test_set, top_k=TOP_K)

# Aggregate core retrieval metrics.
summary = pd.DataFrame({
    "Recall@5": [eval_results["Recall@5"].mean()],
    "MRR": [eval_results["MRR"].mean()]
})

print("Retrieval evaluation summary:")
display(summary)

# Build simple ablation view for presentation and discussion.
ablations = ablation_table(eval_results)
display(ablations)

# Structured output and citation quality checks from one sample output.
if isinstance(structured_output, dict) and "error" not in structured_output:
    citation_grounding = 1.0 if structured_output.get("citations") else 0.0
    structured_accuracy = 1.0
else:
    citation_grounding = 0.0
    structured_accuracy = 0.0

print({
    "Citation Grounding Score": citation_grounding,
    "Structured Output Accuracy": structured_accuracy
})

## Section 10: Streamlit App
The full pipeline is wired in app.py for deployment on Streamlit Community Cloud.

In [ ]:
# Run this command in terminal (not inside notebook):
# streamlit run app.py

print("Streamlit app entrypoint: app.py")
print("UI includes ticker, years, section, spinner, structured output, and raw JSON toggle.")